# 11. PyTorch Tutorial 11 - Softmax and Cross Entropy

In [1]:
import torch
import torch.nn as nn
import numpy as np

### Custom softmax

In [2]:
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=0)

x = np.array([2.0, 1.0, 0.1])
outputs = softmax(x)
print('softmax numpy:', outputs)

softmax numpy: [0.65900114 0.24243297 0.09856589]


### torch softmax

In [3]:
x = torch.tensor([2.0, 1.0, 0.1])
outputs = torch.softmax(x, dim=0) # along values along first axis
print('softmax torch:', outputs)

softmax torch: tensor([0.6590, 0.2424, 0.0986])


### Custom Cross Entropy

`np.clip()` is a function in the NumPy library of Python that is used to limit the elements in an array. The function takes an interval (combination of minimum value and maximum value), and values outside the interval are clipped to the interval edges. For example, if an interval of [0, 1] is specified, values smaller than 0 become 0, and values larger than 1 become 1.

In [4]:
def cross_entropy(actual, predicted):
    EPS = 1e-15 # EPS stands for Epsilon (it helps avoid zero division errors of log errors)
    predicted = np.clip(predicted, EPS, 1 - EPS)
    loss = -np.sum(actual * np.log(predicted)) # element-wise multiplication
    return loss # / float(predicted.shape[0])

#### checking the element-wise multiplication

In [5]:
np.log(np.array([0.7, 0.2, 0.1])) * np.array([1, 0, 0])

array([-0.35667494, -0.        , -0.        ])

 y must be one hot encoded
 
 - if class 0: [1 0 0]
 - if class 1: [0 1 0]
 - if class 2: [0 0 1]

In [6]:
Y = np.array([1, 0, 0])
Y_pred_good = np.array([0.7, 0.2, 0.1])
Y_pred_bad = np.array([0.1, 0.3, 0.6])
l1 = cross_entropy(Y, Y_pred_good)
l2 = cross_entropy(Y, Y_pred_bad)
print(f'Loss1 numpy: {l1:.4f}')
print(f'Loss2 numpy: {l2:.4f}')

Loss1 numpy: 0.3567
Loss2 numpy: 2.3026


### torch Cross Entropy : nn.CrossEntropyLoss()

* The nn.CrossEntropyLoss() objects take in tensor arguments that have'nt been computed by the softmax yet. In fact, the CrossEntropyLoss() uilizes a built-in softmax.

* Even the class we are trying to predict should not be in the `one-hot encoded class`format

In [7]:
loss = nn.CrossEntropyLoss()
# loss(input, target)

# target is of size nSamples = 1
# each element has class label: 0, 1, or 2
# Y (=target) contains class labels, not one-hot
# class 0
Y_0 = torch.tensor([0])
Y_0

*  However the predicted raw values in a 1D vector that get passed in to the loss function
* **input is of size nSamples x nClasses = 1 x 3**
* y_pred (=input) must be raw, unnormalizes scores (logits) for each class, not softmax

In [8]:
# input is of size nSamples x nClasses = 1 x 3
# y_pred (=input) must be raw, unnormalizes scores (logits) for each class, not softmax
Y_pred_good = torch.tensor([[2.0, 1.0, 0.1]])
Y_pred_bad = torch.tensor([[0.5, 2.0, 0.3]])
l1 = loss(Y_pred_good, Y_0)
l2 = loss(Y_pred_bad, Y_0)

print(f'PyTorch Loss1 good: {l1.item():.4f}')
print(f'PyTorch Loss2 bad: {l2.item():.4f}')

PyTorch Loss1 good: 0.4170
PyTorch Loss2 bad: 1.8406


In [9]:
# class 1
Y_1 = torch.tensor([1])
Y_pred_bad = torch.tensor([[2.0, 1.0, 0.1]])
Y_pred_good = torch.tensor([[0.5, 2.0, 0.3]])
l1 = loss(Y_pred_bad, Y_1)
l2 = loss(Y_pred_good, Y_1)

print(f'PyTorch Loss1 bad: {l1.item():.4f}')
print(f'PyTorch Loss2 good: {l2.item():.4f}')

PyTorch Loss1 bad: 1.4170
PyTorch Loss2 good: 0.3406


In [31]:
T = np.random.random(size=9)
T = np.reshape(T, (3, 3))
T = torch.from_numpy(T.astype(np.float64))

### get predictions

This line of code is extracting the class with the maximum probability prediction from a batch of predictions.

Let's break it down:

`Y_pred_good` - This is a tensor of predictions from your model. It has shape (batch_size, num_classes)

`torch.max(Y_pred_good, 1)` - This performs a max operation along the second dimension (axis 1) of the tensor. Since axis 1 corresponds to the class dimension, this gives you the maximum probability prediction for each sample in the batch.

`_, predictions1 = torch.max(Y_pred_good, 1)` - The torch.max() call returns two values: the maximum value, and the index of that maximum value. We use _` to ignore the maximum value, and store only the index in `predictions1`.

So in summary:

- `Y_pred_good` is your model predictions, shape (batch_size, num_classes)
- `torch.max(Y_pred_good, 1)` finds the maximum value along the class axis (axis 1)
- We use `_, predictions1 = ...` to store only the index of the maximum value, not the value itself
- So `predictions1` will contain the index of the predicted class for each sample, with shape (batch_size,)


In [35]:
# max along the row (class dimension)
# this function yields the max value and its index in the row
torch.max(Y_pred_good, 1)

torch.return_types.max(
values=tensor([2.]),
indices=tensor([1]))

In [34]:
# here we are only interested in the index of the max value
_, predictions1 = torch.max(Y_pred_good, 1)
_, predictions2 = torch.max(Y_pred_bad, 1)
print(f'Actual class: {Y_1.item()}, Y_pred1: {predictions1.item()}, Y_pred2: {predictions2.item()}')

Actual class: 1, Y_pred1: 1, Y_pred2: 0


The .item() method on PyTorch tensors is used to retrieve the raw Python value from the tensor.

This is useful for a few reasons:

1. Converting a 1 element tensor to a standard Python type like float, int, etc. This allows you to use the value outside of PyTorch operations.

For example:
```python
t = torch.tensor([1])
print(type(t)) 
# <class 'torch.Tensor'>

value = t.item()  
print(type(value))  
# <class 'int'>
```

2. Accessing a value from a tensor without using tensor operations. This can be more efficient since .item() avoids creating new tensors.

For example:
```python 
t = torch.tensor([1, 2, 3])

value = t[0].item()
print(value)
# 1
```

3. Accessing the single value from a tensor with 1 element.

For example:
```python
t = torch.tensor(3.14)
value = t.item()
print(value)
# 3.14
```

So in short, .item() is useful for extracting the raw Python value from a tensor with 1 element, allowing you to use that value outside of PyTorch operations.

Hope this explanation of .item() helps! Let me know if you have any other questions.

## We can use mutiple samples with the nn.CrossEntropyLoss() function

In [36]:
# allows batch loss for multiple samples

# target is of size nBatch = 3
# each element has class label: 0, 1, or 2
Y = torch.tensor([2, 0, 1])

# input is of size nBatch x nClasses = 3 x 3
# Y_pred are logits (not softmax)
Y_pred_good = torch.tensor(
    [[0.1, 0.2, 3.9], # predict class 2
    [1.2, 0.1, 0.3], # predict class 0
    [0.3, 2.2, 0.2]]) # predict class 1

Y_pred_bad = torch.tensor(
    [[0.9, 0.2, 0.1],
    [0.1, 0.3, 1.5],
    [1.2, 0.2, 0.5]])

In [37]:
# Compute the loss function for the two samples
l1 = loss(Y_pred_good, Y)
l2 = loss(Y_pred_bad, Y)
print(f'Batch Loss1:  {l1.item():.4f}')
print(f'Batch Loss2: {l2.item():.4f}')

Batch Loss1:  0.2834
Batch Loss2: 1.6418


In [38]:
# get predictions
_, predictions1 = torch.max(Y_pred_good, 1)
_, predictions2 = torch.max(Y_pred_bad, 1)
print(f'Actual class: {Y}, Y_pred1: {predictions1}, Y_pred2: {predictions2}')

Actual class: tensor([2, 0, 1]), Y_pred1: tensor([2, 0, 1]), Y_pred2: tensor([0, 2, 0])


## Binary classification

In [39]:
# Binary classification
class NeuralNet1(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(NeuralNet1, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) 
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, 1)  
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # sigmoid at the end
        y_pred = torch.sigmoid(out)
        return y_pred

model = NeuralNet1(input_size=28*28, hidden_size=5)
criterion = nn.BCELoss()

## Multiclass problem

In [ ]:
# Multiclass problem
class NeuralNet2(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet2, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) 
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, num_classes)  
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # no softmax at the end
        return out

model = NeuralNet2(input_size=28*28, hidden_size=5, num_classes=3)
criterion = nn.CrossEntropyLoss()  # (applies Softmax)